# Práctica 1 | Predicción Multiclase de Riesgo Cardiovascular en Población Adulta

## PLANTEAMIENTO DEL PROBLEMA

Las enfermedades cardiovasculares son una de las mayores amenazas para la salud pública global y una de las principales causas de morbimortalidad a nivel mundial.
Para combatir este problema, la detección temprana y la estratificación precisa del nivel de riesgo individual son pilares esenciales que permiten aplicar intervenciones preventivas adaptadas a cada paciente y optimizar la gestión de los recursos en los sistemas de salud.
Conscientes de esta necesidad, el equipo médico e investigador del Hospital Morales Meseguer ha puesto en marcha un proyecto estratégico orientado al desarrollo de herramientas predictivas de vanguardia. 
La iniciativa nace con el objetivo de ofrecer una solución integral basada en datos clínicos que transforme la evaluación del riesgo cardiovascular en la práctica médica diaria, mejorando directamente el diagnóstico y el seguimiento del paciente.
Para ello, el equipo médico pone a disposición de este proyecto una cohorte clínica con 2.254 registros de pacientes y 28 variables sociodemográficas, clínicas, analíticas y de estilo de vida para clasificar a los pacientes en tres niveles de riesgo cardiovascular (Clase 0: Bajo Riesgo, Clase 1: Riesgo Moderado y Clase 2: Alto Riesgo).

## 1. Configuración del entorno y reproducibilidad

Antes de nada, se fijan la **semilla** que se reutilizará en todas las técnicas y las librerías.

**Nota:** Este cuaderno se encuentra dentro del proyecto en `notebooks/`, los datos en `data/` y el código reutilizable en `src/`. Todas las rutas se construyen a partir de la raíz del proyecto que llamremos `RAIZ`.

In [2]:
import random
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn
from scipy import stats

# Importante para que se muestren los gráficos en el cuaderno
%matplotlib inline

# Raíz del proyecto
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# Datos del proyecto
RUTA_DATOS = RAIZ / 'data' / 'raw' / 'enfermedad_cardiovascular.csv'
assert RUTA_DATOS.exists(), f'No se encuentra el dataset en {RUTA_DATOS}'

# Semilla
SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
print('Raíz del proyecto:', RAIZ.name)

Raíz del proyecto: MasterIA-ML01


## 2. Carga de datos

In [ ]:
df = pd.read_csv(RUTA_DATOS)   # data/raw/enfermedad_cardiovascular.csv

print('Cantidad de filas y columnas:', df.shape)
df.head(10)

Cantidad de filas y columnas: (2254, 28)


,edad,presion_sistolica,presion_diastolica,colesterol_total,colesterol_hdl,colesterol_ldl,trigliceridos,glucosa_ayunas,indice_masa_corporal,frecuencia_cardiaca_reposo,...,hemoglobina_glicosilada,fumador_actual,hipertension_previa,diabetes_mellitus,tratamiento_estatinas,uso_antihipertensivos,tipo_alimentacion,nivel_actividad,antecedentes_familiares_tipo,riesgo_cardiovascular
0,57.4,192.0,78.9,371.24,62.6,149.1,245.6,NaN,30.4,95.1,...,5.41,1,1,1,0,1,Mediterránea,Moderadamente activo,Ninguno,2
1,51.7,131.2,80.0,270.20,35.4,NaN,173.8,137.5,29.7,85.3,...,6.11,0,1,0,1,1,Baja en carbohidratos,Ligeramente activo,Ambos,1
2,56.7,127.1,92.5,202.30,47.7,134.1,60.9,157.4,29.4,87.5,...,6.52,1,1,0,0,1,Baja en carbohidratos,Sedentario,Ninguno,2
3,54.9,112.8,97.9,179.20,53.9,137.9,174.4,92.0,28.6,47.4,...,4.06,0,1,0,0,1,Occidental,Ligeramente activo,Paterno,0
4,47.2,115.1,97.0,242.20,56.5,159.2,195.7,63.5,18.5,69.0,...,4.63,0,0,0,0,0,Occidental,Muy activo,Ninguno,0
5,55.5,136.0,76.6,214.50,51.1,134.3,230.7,105.6,28.8,85.5,...,6.38,0,1,1,0,0,Baja en carbohidratos,Sedentario,Paterno,1
6,48.5,115.1,96.4,194.00,48.2,147.5,199.2,92.6,15.4,103.6,...,6.22,0,0,0,0,0,Vegetariana,Sedentario,Ambos,1
7,60.9,132.1,92.4,181.40,41.7,156.0,163.6,98.7,31.2,103.7,...,6.25,1,0,1,1,0,Vegetariana,Sedentario,Ninguno,2
8,52.7,98.3,70.5,213.70,47.0,165.9,246.9,90.4,35.1,87.7,...,6.43,0,1,1,1,1,Occidental,Sedentario,Paterno,1
9,68.1,141.7,88.3,154.70,50.8,133.6,145.2,102.3,33.1,96.2,...,5.92,1,1,0,0,1,Baja en carbohidratos,Ligeramente activo,Paterno,1


In [ ]:
# Configuramos pandas para que muestre todas las columnas sin truncar
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

df.head(10)

,edad,presion_sistolica,presion_diastolica,colesterol_total,colesterol_hdl,colesterol_ldl,trigliceridos,glucosa_ayunas,indice_masa_corporal,frecuencia_cardiaca_reposo,horas_sueno_diarias,consumo_alcohol_semanal,anos_fumador,proteina_c_reactiva,acido_urico,clearance_creatinina,frecuencia_ejercicio_semanal,perimetro_cintura,hemoglobina_glicosilada,fumador_actual,hipertension_previa,diabetes_mellitus,tratamiento_estatinas,uso_antihipertensivos,tipo_alimentacion,nivel_actividad,antecedentes_familiares_tipo,riesgo_cardiovascular
0,57.4,192.0,78.9,371.24,62.6,149.1,245.6,NaN,30.4,95.1,6.5,7.6,9.4,3.80,3.70,69.5,3.3,110.4,5.41,1,1,1,0,1,Mediterránea,Moderadamente activo,Ninguno,2
1,51.7,131.2,80.0,270.20,35.4,NaN,173.8,137.5,29.7,85.3,5.0,8.7,0.0,4.07,4.67,87.5,5.6,88.4,6.11,0,1,0,1,1,Baja en carbohidratos,Ligeramente activo,Ambos,1
2,56.7,127.1,92.5,202.30,47.7,134.1,60.9,157.4,29.4,87.5,8.7,9.7,20.8,3.39,7.15,100.6,1.2,123.8,6.52,1,1,0,0,1,Baja en carbohidratos,Sedentario,Ninguno,2
3,54.9,112.8,97.9,179.20,53.9,137.9,174.4,92.0,28.6,47.4,5.7,5.6,10.0,0.75,4.81,92.0,1.5,93.3,4.06,0,1,0,0,1,Occidental,Ligeramente activo,Paterno,0
4,47.2,115.1,97.0,242.20,56.5,159.2,195.7,63.5,18.5,69.0,6.7,-4.9,5.5,NaN,10.13,93.7,5.7,91.0,4.63,0,0,0,0,0,Occidental,Muy activo,Ninguno,0
5,55.5,136.0,76.6,214.50,51.1,134.3,230.7,105.6,28.8,85.5,NaN,6.4,17.3,2.00,5.61,64.8,1.6,79.7,6.38,0,1,1,0,0,Baja en carbohidratos,Sedentario,Paterno,1
6,48.5,115.1,96.4,194.00,48.2,147.5,199.2,92.6,15.4,103.6,8.0,0.3,0.0,4.11,7.25,78.1,5.9,95.7,6.22,0,0,0,0,0,Vegetariana,Sedentario,Ambos,1
7,60.9,132.1,92.4,181.40,41.7,156.0,163.6,98.7,31.2,103.7,NaN,5.9,6.1,1.85,6.88,94.1,2.6,124.3,6.25,1,0,1,1,0,Vegetariana,Sedentario,Ninguno,2
8,52.7,98.3,70.5,213.70,47.0,165.9,246.9,90.4,35.1,87.7,5.8,8.7,15.1,3.40,5.80,73.8,2.9,93.9,6.43,0,1,1,1,1,Occidental,Sedentario,Paterno,1
9,68.1,141.7,88.3,154.70,50.8,133.6,145.2,102.3,33.1,96.2,6.1,1.0,24.4,1.23,5.06,71.4,3.9,102.8,5.92,1,1,0,0,1,Baja en carbohidratos,Ligeramente activo,Paterno,1


In [7]:
# Analizamos algunos estadísticos asociados a los datos.
df.describe()

,edad,presion_sistolica,presion_diastolica,colesterol_total,colesterol_hdl,colesterol_ldl,trigliceridos,glucosa_ayunas,indice_masa_corporal,frecuencia_cardiaca_reposo,horas_sueno_diarias,consumo_alcohol_semanal,anos_fumador,proteina_c_reactiva,acido_urico,clearance_creatinina,frecuencia_ejercicio_semanal,perimetro_cintura,hemoglobina_glicosilada,fumador_actual,hipertension_previa,diabetes_mellitus,tratamiento_estatinas,uso_antihipertensivos,riesgo_cardiovascular
count,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000,2119.000000,2254.000000,2142.000000,2254.000000,2254.000000,2074.000000,2254.000000,2254.000000,2097.000000,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000,2254.000000
mean,56.688287,132.315186,82.167524,228.133829,47.225200,135.638603,176.657391,104.107330,28.093390,73.668190,6.750916,4.129193,7.465129,2.347864,6.057396,86.631766,2.778305,91.474889,5.807489,0.263975,0.366903,0.227595,0.271961,0.349157,0.700089
std,14.448018,26.931005,13.084058,62.725948,15.991614,36.732756,72.735893,24.939696,6.470585,12.762546,1.583429,2.985199,6.670252,1.969701,1.686887,18.374585,1.874127,14.874715,0.840066,0.440884,0.482067,0.419373,0.445069,0.476809,0.781289
min,9.800000,52.200000,30.800000,61.100000,-8.800000,6.100000,-35.700000,32.000000,3.600000,25.800000,1.600000,-5.000000,0.000000,0.000000,2.140000,31.200000,0.000000,44.200000,2.820000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,47.200000,115.300000,73.600000,188.200000,36.700000,111.400000,133.050000,86.300000,23.800000,65.000000,5.700000,2.200000,0.800000,0.950000,4.990000,74.300000,1.300000,81.400000,5.250000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,56.900000,129.650000,82.100000,221.200000,47.300000,135.500000,170.000000,103.750000,28.000000,73.100000,6.800000,4.000000,6.500000,1.790000,5.890000,86.700000,2.800000,91.300000,5.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000
75%,66.475000,143.900000,91.100000,252.500000,58.200000,161.400000,208.100000,120.475000,32.500000,82.300000,7.800000,6.100000,12.100000,3.230000,6.770000,98.975000,4.000000,101.500000,6.380000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000
max,103.500000,231.360000,122.500000,448.590000,106.300000,266.700000,452.700000,194.000000,50.600000,112.700000,12.100000,15.300000,39.900000,12.880000,12.290000,147.800000,10.000000,148.900000,8.780000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000


In [5]:
# 2. (Opcional pero muy recomendable) Transponer el resultado con .T
# Esto coloca las estadísticas en columnas y tus 28 variables hacia abajo,
# facilitando mucho la lectura visual sin desplazamiento horizontal.
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
edad,2254.0,56.688287,14.448018,9.80,47.20,56.90,66.475,103.50
presion_sistolica,2254.0,132.315186,26.931005,52.20,115.30,129.65,143.900,231.36
presion_diastolica,2254.0,82.167524,13.084058,30.80,73.60,82.10,91.100,122.50
colesterol_total,2254.0,228.133829,62.725948,61.10,188.20,221.20,252.500,448.59
colesterol_hdl,2254.0,47.225200,15.991614,-8.80,36.70,47.30,58.200,106.30
colesterol_ldl,2119.0,135.638603,36.732756,6.10,111.40,135.50,161.400,266.70
trigliceridos,2254.0,176.657391,72.735893,-35.70,133.05,170.00,208.100,452.70
glucosa_ayunas,2142.0,104.107330,24.939696,32.00,86.30,103.75,120.475,194.00
indice_masa_corporal,2254.0,28.093390,6.470585,3.60,23.80,28.00,32.500,50.60
frecuencia_cardiaca_reposo,2254.0,73.668190,12.762546,25.80,65.00,73.10,82.300,112.70


El fichero tiene **2.254 registros y 28 columnas**: 27 predictores y la variable objetivo. Se puede apreciar:
* Hay valores ausentes (`NaN`). 
* Tambien es de resaltar que la edad tiene decimales aunque en las instrucciones dice *años cumplidos*.
* Vemos que `colesterol_hdl` y `trigliceridos` tienen valores negativos cuando deberían ser todos positivos.

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2254 entries, 0 to 2253
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   edad                          2254 non-null   float64
 1   presion_sistolica             2254 non-null   float64
 2   presion_diastolica            2254 non-null   float64
 3   colesterol_total              2254 non-null   float64
 4   colesterol_hdl                2254 non-null   float64
 5   colesterol_ldl                2119 non-null   float64
 6   trigliceridos                 2254 non-null   float64
 7   glucosa_ayunas                2142 non-null   float64
 8   indice_masa_corporal          2254 non-null   float64
 9   frecuencia_cardiaca_reposo    2254 non-null   float64
 10  horas_sueno_diarias           2074 non-null   float64
 11  consumo_alcohol_semanal       2254 non-null   float64
 12  anos_fumador                  2254 non-null   float64
 13  pro

El dataset consta de 28 atributos en total: 1 variable objetivo (target) y 27 variables de entrada (predictores).

### Variable Objetivo (Target)
*	**riesgo_cardiovascular**: Variable entera (int64) que clasifica a cada paciente según tres niveles de riesgo:
  *	0: Bajo riesgo.
  * 1: Riesgo moderado.
  * 2: Alto riesgo.


### Variables de Entrada (27 Predictores)
#### Variables Numéricas `float64` (19)
* Desde `0 edad` hasta `18 hemoglobina_glicosilada`.

#### Variables Booleanas / Binarias `int64` (5) (Valores 1: Sí, 0: No)
*	fumador_actual, hipertension_previa, diabetes_mellitus *(diabetes  tipo 2)*, tratamiento_estatinas, uso_antihipertensivos

#### Variables Nominales / Categóricas `object` (3)
*	tipo_alimentacion: Mediterránea, Occidental, Vegetariana, Baja en carbohidratos.
*	nivel_actividad: Sedentario, Ligeramente activo, Moderadamente activo, Muy activo.
*	antecedentes_familiares_tipo: Ninguno, Materno, Paterno, Ambos.

Para no repetir listas de columnas en todo el cuaderno, definimos aquí los grupos de variables.

In [9]:
TARGET = 'riesgo_cardiovascular'

COLS_NUMERICAS = [
    'edad', 'presion_sistolica', 'presion_diastolica', 'colesterol_total',
    'colesterol_hdl', 'colesterol_ldl', 'trigliceridos', 'glucosa_ayunas',
    'indice_masa_corporal', 'frecuencia_cardiaca_reposo', 'horas_sueno_diarias',
    'consumo_alcohol_semanal', 'anos_fumador', 'proteina_c_reactiva',
    'acido_urico', 'clearance_creatinina', 'frecuencia_ejercicio_semanal',
    'perimetro_cintura', 'hemoglobina_glicosilada',
]
COLS_BOOLEANAS = [
    'fumador_actual', 'hipertension_previa', 'diabetes_mellitus',
    'tratamiento_estatinas', 'uso_antihipertensivos',
]
COLS_NOMINALES = ['tipo_alimentacion', 'nivel_actividad', 'antecedentes_familiares_tipo']

# Comprobación: los grupos y el target cubren exactamente las columnas del CSV
print(f'{len(COLS_NUMERICAS)} numéricas, {len(COLS_BOOLEANAS)} booleanas, '
      f'{len(COLS_NOMINALES)} nominales y 1 objetivo')

df.info()

19 numéricas, 5 booleanas, 3 nominales y 1 objetivo
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2254 entries, 0 to 2253
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   edad                          2254 non-null   float64
 1   presion_sistolica             2254 non-null   float64
 2   presion_diastolica            2254 non-null   float64
 3   colesterol_total              2254 non-null   float64
 4   colesterol_hdl                2254 non-null   float64
 5   colesterol_ldl                2119 non-null   float64
 6   trigliceridos                 2254 non-null   float64
 7   glucosa_ayunas                2142 non-null   float64
 8   indice_masa_corporal          2254 non-null   float64
 9   frecuencia_cardiaca_reposo    2254 non-null   float64
 10  horas_sueno_diarias           2074 non-null   float64
 11  consumo_alcohol_semanal       2254 non-null   float64
 12  anos_fumad

Los tipos son coherentes con la información del enunciado: hay 19 numéricas (`float64`), las booleanas y el target son `int64` y las nominales son texto (`object`). Si una columna numérica hubiera salido como texto (`object`), sería señal de que contiene algo raro, como etiquetas o códigos de error. El recuento de no nulos ya delata **cuatro columnas con ausentes**: `colesterol_ldl`, `glucosa_ayunas`, `horas_sueno_diarias` y `proteina_c_reactiva`.

In [10]:
df.isnull().sum()

edad                              0
presion_sistolica                 0
presion_diastolica                0
colesterol_total                  0
colesterol_hdl                    0
colesterol_ldl                  135
trigliceridos                     0
glucosa_ayunas                  112
indice_masa_corporal              0
frecuencia_cardiaca_reposo        0
horas_sueno_diarias             180
consumo_alcohol_semanal           0
anos_fumador                      0
proteina_c_reactiva             157
acido_urico                       0
clearance_creatinina              0
frecuencia_ejercicio_semanal      0
perimetro_cintura                 0
hemoglobina_glicosilada           0
fumador_actual                    0
hipertension_previa               0
diabetes_mellitus                 0
tratamiento_estatinas             0
uso_antihipertensivos             0
tipo_alimentacion                 0
nivel_actividad                   0
antecedentes_familiares_tipo      0
riesgo_cardiovascular       

In [11]:
df.duplicated().sum()

np.int64(0)

Con esto, podemos empezar a ver los errores en los datos. Destacamos:

1. Para la variable colesterol_ldl, faltan 135 registros.
2. Para la variable glucosa_ayunas tenemos 112 nulos.
3. La variable horas_sueno_diarias tiene 180 nulos.
4. La variable proteina_c_reactiva tiene 157 nulos.
5. Para la variables colesterol_hdl o trigliceridos, hay valores negativos, algo que es biológicamente imposible.

Aquí ya hay que empezar a tomar decisiones, sobre cómo abordar estos fallos en los datos.

Para las variables que contienen tantos nulos, vamos a ver cómo se relacionan con la variable objetivo.

In [ ]:
import seaborn as sns

variables_clave = ['colesterol_ldl', 'glucosa_ayunas', 'horas_sueno_diarias', 'proteina_c_reactiva', 'riesgo_cardiovascular']

sns.pairplot(df[variables_clave], hue="riesgo_cardiovascular", height = 2, palette = 'colorblind')

Antes de tomar alguna decisión con respecto a los nulos, vamos a ver qué porcentaje representan sobre el total.

In [ ]:
df.isnull().mean().sort_values(ascending=False) * 100

In [ ]:
import seaborn as sns

variables_clave = ['colesterol_ldl', 'glucosa_ayunas', 
                   'horas_sueno_diarias', 'proteina_c_reactiva', 'riesgo_cardiovascular']

plt.figure(figsize=(14, 16))
sns.boxplot(data=df[variables_clave])
plt.tight_layout()
plt.show()

Dado que los porcentajes de nulos no son muy altos (entre un 5 y un 8%) y que dos de las variables tienen valores outlier, vamos a optar por usar la mediana para imputar los valores ausentes. 

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
df[variables_clave] = imputer.fit_transform(df[variables_clave])

In [ ]:
df.isnull().mean().sort_values(ascending=False) * 100

In [ ]:
import seaborn as sns

variables_clave = ['colesterol_ldl', 'glucosa_ayunas', 
                   'horas_sueno_diarias', 'proteina_c_reactiva', 'riesgo_cardiovascular']

plt.figure(figsize=(14, 16))
sns.boxplot(data=df[variables_clave])
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

variables_clave = ['colesterol_ldl', 'glucosa_ayunas', 'horas_sueno_diarias', 'proteina_c_reactiva', 'riesgo_cardiovascular']

sns.pairplot(df[variables_clave], hue="riesgo_cardiovascular", height = 2, palette = 'colorblind')

Una vez que hemos hecho estos cambios, no se aprecian variaciones significativas en la dispersión de las clases.

El siguiente paso es comprobar cuántos valores outliers tenemos para cada variable.

In [ ]:
for c in df.columns:
    if df[c].dtype in ['int64', 'float64']:
        Q1 = df[c].quantile(0.25)
        Q3 = df[c].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[c] < lower_bound) | (df[c] > upper_bound)]
        print(f"Variable: {c}, Outliers: {len(outliers)}")

Vamos a ver cómo se relacionan las variables con muchos outliers con la variable objetivo.

In [ ]:
import seaborn as sns

#variables_clave = ['diabetes_mellitus', 'acido_urico', 'proteina_c_reactiva', 'colesterol_total', 'presion_sistolica', 'trigliceridos', 'riesgo_cardiovascular']
variables_clave = ['acido_urico', 'proteina_c_reactiva', 'colesterol_total', 'presion_sistolica', 'trigliceridos', 'riesgo_cardiovascular']

sns.pairplot(df[variables_clave], hue="riesgo_cardiovascular", height = 2, palette = 'colorblind')

- diabetes_mellitus (Booleano): Diagnóstico previo de diabetes mellitus tipo 2 (1: Sí, 0: No).

Esta variable es la que contiene más outliers, pero en realidad, el error es provocado porque es un valor booleano, que solo puede tomar 0's o 1's. Vamos a cambiar el tipo de dato de la variable a booleano, para dejar de tener en cuenta esos "supuestos" valores outliers.

In [ ]:
df['diabetes_mellitus'] = df['diabetes_mellitus'].astype(bool)

In [ ]:
df['diabetes_mellitus'].dtypes

Para el resto de variables con muchos valores 'outliers', vamos a analizar sus valores, para sabe si se trata de errores, o si por el contrario, son pacientes con esos valores muy altos, pero dentro de la normalidad.

In [ ]:
# presion_sistolica, colesterol_total, trigliceridos, proteina_c_reactiva, acido_urico

df[variables_clave].agg(['min', 'max'])

1. acido_urico --> El valor máximo de 12.29 es fisiológicamente posible, por lo que no lo vamos a eliminar.

Rangos normales de referencia
- Hombres: 3.4 - 7.0 
- Mujeres: 2.4 - 6.0 
- Niños / Población pediátrica: 2.0 - 5.5 

2. proteína_c_reactiva --> El valor máximo de 12,88 se considera clínicamente posible, por ello, tampoco lo vamos a eliminar. La Proteína C Reactiva es un reactante de fase aguda que aumenta solo cuando hay procesos inflamatorios, infecciones o daño tisular. Un valor de 0 mg/L (o cercano a cero) indica ausencia de inflamación sistémica

3. colesterol_total --> También se consideran valores normales.

4. presion_sistolica --> Mantenemos los valores.

- Mínimo: 52.20
    - Evaluación: Es una cifra de hipotensión muy marcada (extrema), pero los valores en la cola inferior aumentan gradualmente (62.8, 63.5, 65.4, 70.0, etc.), representando pacientes con presión arterial muy baja.
- Máximo: 231.36 
    - Evaluación: Está truncado (Top-Coded). Al igual que el ácido úrico y el colesterol total, el valor exacto de 231.36 se repite en 53 pacientes diferentes, lo que confirma que las lecturas por encima de ese umbral se acotaron a 231.36.

5. trigliceridos

- Aquí sí tenemos un problema. El valor mínimo debe estar entre 30-40 y tenemos valores negativos.

- La solución que hemos decidido, es convertir los valores inferiores a 30 a nulos, y después imputarles la mediana. 

In [ ]:
import numpy as np

# Umbral fisiológico mínimo para triglicéridos (40 mg/dL)
UMBRAL_MIN_TRIGLICERIDOS = 30.0

df['trigliceridos'] = df['trigliceridos'].apply(lambda x: np.nan if x < UMBRAL_MIN_TRIGLICERIDOS else x)

df['trigliceridos'].describe()

In [ ]:
# Imputamos ahora los valores faltantes de la variable 'trigliceridos' con la mediana de la misma.
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
df[variables_clave] = imputer.fit_transform(df[variables_clave])

df['trigliceridos'].describe()

- Ahora se me genera la duda, de si deberíamos hacer lo que hemos hecho con los trigliceridos, para cualquier variable que tome valores "imposibles".

In [ ]:

variables_con_valores_negativos = ['colesterol_hdl', 'consumo_alcohol_semanal']
df[variables_con_valores_negativos].describe()

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer

# Umbral fisiológico mínimo para COLESTEROL HDL (20 mg/dL)
UMBRAL_MIN_COLESTEROL_HDL = 20.0
UMBRAL_MIN_CONSUMO_ALCOHOL_SEMANAL = 0.0

df['colesterol_hdl'] = df['colesterol_hdl'].apply(lambda x: np.nan if x < UMBRAL_MIN_COLESTEROL_HDL else x)
df['consumo_alcohol_semanal'] = df['consumo_alcohol_semanal'].apply(lambda x: np.nan if x < UMBRAL_MIN_CONSUMO_ALCOHOL_SEMANAL else x)

imputer = SimpleImputer(strategy='median')
df[variables_con_valores_negativos] = imputer.fit_transform(df[variables_con_valores_negativos])

df[variables_con_valores_negativos].describe()


Vamos a revisar si las clases de la variable objetivo están desbalanceadas.

In [ ]:
# Conteo absoluto de cada clase
conteo = df['riesgo_cardiovascular'].value_counts()

# Porcentaje relativo de cada clase
porcentaje = df['riesgo_cardiovascular'].value_counts(normalize=True) * 100

# Combinar en una tabla
df_desbalance = pd.DataFrame({'Cantidad': conteo, 'Porcentaje (%)': porcentaje})
print(df_desbalance)

Como se observa, las clases están desbalanceadas, ya que si por ejemplo, creamos un modelo que siempre diga SIEMPRE que hay un "riesgo bajo" de enfermedad cardiovascular, acertaría en el 50% de los casos.

El siguiente paso va a ser analizar los tipos de datos asociados a cada una de las variables.

In [ ]:
df.dtypes

Aquí podemos ver que hay 3 variables categóricas: tipo_alimentación, nivel_actividad y antecedentes_familiares_tipo. 

Habrá que decidir qué hacer con ellas, dummies o OneHotEncoding.

In [ ]:
# Vamos a ver los valores que almacenan las 3 variables categóricas.

variables = ['tipo_alimentacion','nivel_actividad','antecedentes_familiares_tipo']

for col in df[variables].columns:
    valores = df[col].unique()
    print(f"Columna: {col} ({len(valores)} valores distintos)")
    print(valores)
    print("-" * 40)

Vamos a convertir las variables categóricas en Dummies.

In [ ]:
categoricas = df.select_dtypes(include = ["object", "category"]).columns
# Mostramos las columnas extraídas
categoricas

In [ ]:
# No eliminamos ninguna de las opciones --> drop_first = False.
# Creamos una columna para los nulos --> dummy_na = True.
categoricas_Datos = pd.get_dummies(df[categoricas], drop_first = False, dummy_na = True)
# Mostramos la transformación de las variables categóricas en variables dummies
categoricas_Datos.head()

In [ ]:
df.drop(categoricas, axis = 1, inplace = True)
# Tras eliminar los atributos originales, concatenamos los nuevos atributos creados para las variables categóricas.
datos = pd.concat([df, categoricas_Datos], axis = 1)
# Mostramos como quedan los datos de nuevo.
datos.head()

Teníamos originalmente 28 columnas. Hemos eliminado las 3 originales y hemos añadido 5 (4 + nulos) más por cada variable categórica --> 28-3+16 = 41. Obtenemos 40 porque parece que no hay nadie con dieta baja en carbohidratos.

In [ ]:
# Hacemos de nuevo dtype() para ver cuál es el estado actual.

datos.dtypes

Ahora ya solo tenemos variables int, float y booleanos.

In [ ]:
#(Clase 0: Bajo Riesgo, Clase 1: Riesgo Moderado y Clase 2: Alto Riesgo)

# Elegimos 3 variables para evitar que el gráfico sea inmanejable.
import seaborn as sns

variables_clave = ['edad', 'presion_sistolica', 'colesterol_total', 'riesgo_cardiovascular']

sns.pairplot(df[variables_clave], hue="riesgo_cardiovascular", height = 2, palette = 'colorblind')


El siguiente paso va a ser ejecutar bloxpot() para analizar la dispersión de los datos.

In [ ]:
import seaborn as sns

plt.figure(figsize=(14, 16))
sns.boxplot(data=datos, orient='h')
plt.tight_layout()
plt.show()


Analizando el gráfico encontramos rápidamente dos situaciones: 

1. Hay mucha diferencia en los rangos de valores entre algunas variables. Esto nos obligará a normalizar.

In [ ]:
# Importamos la clase StandardScaler para realizar el escalado
from sklearn.preprocessing import StandardScaler
# Importamos la clase LinearSVC
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split

# Instanciamos el escalador
scaler = StandardScaler()

In [ ]:
X=datos.drop(columns=["riesgo_cardiovascular"])
y=datos["riesgo_cardiovascular"]
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size=0.2,random_state=0)

X_ent_escaladas = pd.DataFrame(scaler.fit_transform(X_train),columns=X_train.columns)

X_test_escaladas = pd.DataFrame(scaler.transform(X_test),columns=X_test.columns)

print(X_ent_escaladas)

# ¿¿Funcionaaa??? ¿¿Puedo visualizar algo??
#X_ent_escaladas.boxplot()

In [ ]:
plt.figure(figsize=(14, 16))
sns.boxplot(data=X_ent_escaladas, orient='h')
plt.tight_layout()
plt.show()

Hemos conseguido que todas las variables estén entre en el rango [-4,4], aunque aún vemos que hay valores outlier.